# 01 — Daily weather for the municipalities of Paraná (BR-DWGD, Xavier et al.)

Extracts, for the seat of each of the 399 municipalities of Paraná, the daily series from 1998-12-01 to
2025-12-31 of the **Brazilian Daily Weather Gridded Data** (BR-DWGD v3.2.4, 0.1° grid;
Xavier et al., 2022, *Int. J. Climatol.* 42:8390–8404), published on Google Earth Engine by its author
(`projects/ee-alexandrexavier/assets/BR-DWGD`).

Variables: precipitation (`pr`, mm), maximum and minimum temperature (`Tmax`, `Tmin`, °C), relative
humidity (`RH`, %), global solar radiation (`Rs`, MJ m⁻² d⁻¹), wind speed at 2 m (`u2`, m s⁻¹) and
reference evapotranspiration (`ET`, mm, Penman-Monteith FAO-56). The asset stores integers, converted
with the `BAND_*_MULT`/`BAND_*_ADD` factors recorded in the properties of each image.

Requires an Earth Engine account with a registered Cloud project (`PROJETO_EE`). Takes ~10–20 min.

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import ee
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DADOS = RAIZ / "dados"
SAIDA = DADOS / "xavier_diario_pr.csv.gz"

PROJETO_EE = "mapbiomas-brazil"
ASSET = "projects/ee-alexandrexavier/assets/BR-DWGD"
INICIO, FIM = "1998-12-01", "2026-01-01"  # FIM is exclusive
BANDAS = ["pr", "Tmax", "Tmin", "RH", "Rs", "u2", "ET"]

ee.Initialize(project=PROJETO_EE)
colecao = ee.ImageCollection(ASSET)
props = colecao.first().toDictionary().getInfo()
ESCALA = {b: (props[f"BAND_{b}_MULT"], props[f"BAND_{b}_ADD"]) for b in BANDAS}
ESCALA

/Users/coted-sh/Python/ML_milho_oeste/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/coted-sh/Python/ML_milho_oeste/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


/Users/coted-sh/Python/ML_milho_oeste/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/coted-sh/Python/ML_milho_oeste/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


{'pr': (0.00686666, 225),
 'Tmax': (0.00106815, 15),
 'Tmin': (0.00106815, 15),
 'RH': (0.39370079, -0.393701),
 'Rs': (0.15708661, -0.057087),
 'u2': (0.05905512, -0.059055),
 'ET': (0.0511811, -0.0511811)}

In [2]:
municipios = pd.read_csv(DADOS / "municipios_pr.csv")
pontos = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r.longitude, r.latitude]), {"cod_ibge": int(r.cod_ibge)})
    for r in municipios.itertuples()
])
projecao = colecao.first().select("pr").projection()


def extrair_periodo(inicio, fim):
    """Stacks the days of the period as bands and samples the 399 points at once."""
    imgs = colecao.filterDate(inicio, fim).select(BANDAS)
    datas = imgs.aggregate_array("system:index").getInfo()
    pilha = imgs.toBands()
    amostras = pilha.sampleRegions(collection=pontos, properties=["cod_ibge"], projection=projecao,
                                   geometries=False).getInfo()["features"]
    linhas = []
    for f in amostras:
        p = f["properties"]
        for i, dia in enumerate(datas):
            reg = {"cod_ibge": p["cod_ibge"], "data": dia[-8:]}
            for b in BANDAS:
                v = p.get(f"{dia}_{b}")
                mult, add = ESCALA[b]
                reg[b] = None if v is None else v * mult + add
            linhas.append(reg)
    return pd.DataFrame(linhas)


periodos = pd.date_range(INICIO, FIM, freq="MS")
periodos = [(a.strftime("%Y-%m-%d"), b.strftime("%Y-%m-%d")) for a, b in zip(periodos[:-1], periodos[1:])]
print(len(periodos), "months to extract")

325 meses a extrair


In [3]:
partes, falhas = [], []
with ThreadPoolExecutor(max_workers=8) as ex:
    futuros = {ex.submit(extrair_periodo, a, b): (a, b) for a, b in periodos}
    for fut in as_completed(futuros):
        try:
            partes.append(fut.result())
        except Exception as erro:  # quota/timeout: re-extracted below, serially
            falhas.append(futuros[fut])

for a, b in falhas:
    partes.append(extrair_periodo(a, b))

clima = pd.concat(partes, ignore_index=True)
clima["data"] = pd.to_datetime(clima["data"], format="%Y%m%d")
clima = clima.sort_values(["cod_ibge", "data"]).reset_index(drop=True)
print(f"{len(falhas)} months re-extracted; {clima['cod_ibge'].nunique()} municipalities; "
      f"{clima['data'].min().date()} to {clima['data'].max().date()}")

0 meses reextraídos; 399 municípios; 1998-12-01 a 2025-12-31


## Quality control

Seats very close to the coast may fall in a cell without data; in that case the mean of the nearest
neighbouring municipalities (up to 3) on the same day is used.

In [4]:
faltantes = clima[BANDAS].isna().groupby(clima["cod_ibge"]).mean()
sem_dado = faltantes.index[faltantes.max(axis=1) > 0.5].tolist()
print("Municipalities without data (>50% missing):", sem_dado)

if sem_dado:
    import numpy as np
    coords = municipios.set_index("cod_ibge")[["latitude", "longitude"]]
    completos = coords.drop(index=sem_dado)
    piv = {b: clima.pivot(index="data", columns="cod_ibge", values=b) for b in BANDAS}
    for cod in sem_dado:
        dist = np.hypot(*(completos - coords.loc[cod]).T.values)
        vizinhos = completos.index[np.argsort(dist)[:3]]
        for b in BANDAS:
            piv[b][cod] = piv[b][vizinhos].mean(axis=1)
    clima = pd.concat({b: p.stack() for b, p in piv.items()}, axis=1).reset_index()
    clima.columns = ["data", "cod_ibge"] + BANDAS

print(clima[BANDAS].describe().round(2).to_string())
clima.round(3).to_csv(SAIDA, index=False, compression="gzip")
print("saved:", SAIDA, f"{SAIDA.stat().st_size / 1e6:.1f} MB")

Municípios sem dado (>50% faltante): []


               pr        Tmax        Tmin          RH          Rs          u2          ET
count  3947307.00  3947307.00  3947307.00  3947307.00  3947307.00  3947307.00  3947307.00
mean         4.45       27.33       16.25       74.07       17.04        1.49        3.52
std         10.19        4.89        4.46       11.96        6.39        0.62        1.48
min          0.00        4.69       -8.12       22.83        0.73        0.00        0.41
25%          0.00       24.28       13.66       66.14       12.67        1.06        2.35
50%          0.00       27.97       16.93       75.20       16.75        1.36        3.43
75%          3.76       30.89       19.60       83.07       21.62        1.83        4.66
max        233.53       42.46       28.20       99.21       33.87        5.55        9.62


salvo: /Users/coted-sh/Python/ML_milho_oeste/dados/xavier_diario_pr.csv.gz 72.2 MB
